In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from pandas.api.types import CategoricalDtype

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else (cwd.parent if cwd.name == "src" else cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUTA_RAW = ROOT / "Data" / "raw" / "hotel_bookings.csv"
RUTA_ML = ROOT / "Data" / "ml"
RUTA_ENCODERS = ROOT / "models" / "encoders"
RUTA_ML.mkdir(parents=True, exist_ok=True)
RUTA_ENCODERS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
ORDEN_MESES = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

In [2]:
df = pd.read_csv(RUTA_RAW)

# Drop columnas con data leakage (se conocen tras la cancelación)
df = df.drop(columns=["reservation_status", "reservation_status_date"])

# Tratamiento de nulos
df["children"] = df["children"].fillna(0).astype("int64")
df["country"] = df["country"].fillna("desconocido")
df["agent"] = df["agent"].fillna(0).astype("int64")
df["has_company"] = df["company"].notna().astype("int8")
df = df.drop(columns=["company"])

# Tipos
df["arrival_date_month"] = df["arrival_date_month"].astype(
    CategoricalDtype(categories=ORDEN_MESES, ordered=True)
)
df["is_repeated_guest"] = df["is_repeated_guest"].astype("bool")

print(df.shape, "| nulos:", df.isna().sum().sum())
df.head()

(119390, 30) | nulos: 0


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_company
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,C,3,No Deposit,0,0,Transient,0.0,0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,C,4,No Deposit,0,0,Transient,0.0,0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,C,0,No Deposit,0,0,Transient,75.0,0,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,A,0,No Deposit,304,0,Transient,75.0,0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,0,No Deposit,240,0,Transient,98.0,0,1,0


In [3]:
meses_a_num = {m: i for i, m in enumerate(ORDEN_MESES, start=1)}
mes_num = df["arrival_date_month"].map(meses_a_num).astype("int64")

arrival_date = pd.to_datetime({
    "year": df["arrival_date_year"],
    "month": mes_num,
    "day": df["arrival_date_day_of_month"],
})

df["day_of_week"] = arrival_date.dt.dayofweek.astype("int8")
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype("int8")
df["month_sin"] = np.sin(2 * np.pi * mes_num / 12)
df["month_cos"] = np.cos(2 * np.pi * mes_num / 12)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

df.shape

(119390, 38)

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["is_canceled"])
y = df["is_canceled"].astype("int8")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp,
)

for nombre, y_s in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{nombre:>5}: {len(y_s):>6} filas | ratio is_canceled={y_s.mean():.4f}")

train:  83573 filas | ratio is_canceled=0.3704
  val:  17908 filas | ratio is_canceled=0.3704
 test:  17909 filas | ratio is_canceled=0.3704


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from src.preprocesado_utils import WinsorizerPercentil

COLS_OHE = [
    "hotel", "meal", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type",
]
COLS_TARGET = ["country", "agent"]
COLS_WINSOR = ["lead_time", "adr", "days_in_waiting_list"]

encoders = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), COLS_OHE),
    ("ord", OrdinalEncoder(categories=[ORDEN_MESES], handle_unknown="use_encoded_value", unknown_value=-1), ["arrival_date_month"]),
    ("tgt", TargetEncoder(target_type="binary", random_state=RANDOM_STATE), COLS_TARGET),
], remainder="passthrough", verbose_feature_names_out=False)

preprocesador = Pipeline([
    ("winsor", WinsorizerPercentil(columnas=COLS_WINSOR)),
    ("cols", encoders),
])
preprocesador.fit(X_train, y_train)

feature_names = preprocesador.named_steps["cols"].get_feature_names_out().tolist()

def transformar(X_df):
    arr = np.asarray(preprocesador.transform(X_df), dtype="float64")
    return pd.DataFrame(arr, columns=feature_names, index=X_df.index)

X_train_enc = transformar(X_train)
X_val_enc = transformar(X_val)
X_test_enc = transformar(X_test)

X_train_enc.to_csv(RUTA_ML / "X_train.csv", index=False)
X_val_enc.to_csv(RUTA_ML / "X_val.csv", index=False)
X_test_enc.to_csv(RUTA_ML / "X_test.csv", index=False)
y_train.to_frame("is_canceled").to_csv(RUTA_ML / "y_train.csv", index=False)
y_val.to_frame("is_canceled").to_csv(RUTA_ML / "y_val.csv", index=False)
y_test.to_frame("is_canceled").to_csv(RUTA_ML / "y_test.csv", index=False)

joblib.dump(preprocesador, RUTA_ENCODERS / "preprocesador.joblib")
with open(RUTA_ENCODERS / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

print(f"X_train_enc: {X_train_enc.shape} | {len(feature_names)} features")

X_train_enc: (83573, 70) | 70 features


In [6]:
import joblib
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

modelo = LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
modelo.fit(X_train_enc, y_train)

for split, X_s, y_s in [
    ("TRAIN", X_train_enc, y_train),
    ("VAL", X_val_enc, y_val),
    ("TEST", X_test_enc, y_test),
]:
    y_pred = modelo.predict(X_s)
    y_proba = modelo.predict_proba(X_s)[:, 1]
    print(f"\n=== {split} ===")
    print(f"Accuracy: {accuracy_score(y_s, y_pred):.4f}")
    print(f"F1:       {f1_score(y_s, y_pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y_s, y_proba):.4f}")
    print(classification_report(y_s, y_pred, digits=4))
    print(confusion_matrix(y_s, y_pred))

RUTA_MODELO = ROOT / "models" / "clasificador.joblib"
joblib.dump(modelo, RUTA_MODELO)
print(f"\nGuardado: {RUTA_MODELO}")


=== TRAIN ===
Accuracy: 0.8839
F1:       0.8387
ROC-AUC:  0.9557
              precision    recall  f1-score   support

           0     0.8944    0.9249    0.9094     52616
           1     0.8644    0.8144    0.8387     30957

    accuracy                         0.8839     83573
   macro avg     0.8794    0.8696    0.8740     83573
weighted avg     0.8833    0.8839    0.8832     83573

[[48662  3954]
 [ 5746 25211]]

=== VAL ===
Accuracy: 0.8739
F1:       0.8235
ROC-AUC:  0.9487
              precision    recall  f1-score   support

           0     0.8837    0.9210    0.9019     11275
           1     0.8553    0.7939    0.8235      6633

    accuracy                         0.8739     17908
   macro avg     0.8695    0.8574    0.8627     17908
weighted avg     0.8732    0.8739    0.8729     17908

[[10384   891]
 [ 1367  5266]]

=== TEST ===
Accuracy: 0.8761
F1:       0.8272
ROC-AUC:  0.9485
              precision    recall  f1-score   support

           0     0.8869    0.9205 